# Phase 3: Enterprise Data Warehouse Design

## Objective

The purpose of this phase is to transform the raw Olist operational database into an enterprise-ready analytical data model.

Instead of querying multiple transactional tables directly, we will design a dimensional model that supports fast reporting, executive dashboards, business intelligence, and machine learning workloads.

The warehouse will follow Kimball's dimensional modeling principles using a Star Schema architecture.

# Why Build a Data Warehouse?

The raw Olist dataset was designed for transactional operations.

While suitable for processing orders, it is not optimized for analytical workloads because:

- Data is spread across multiple tables.
- Business metrics require numerous joins.
- Query performance decreases as data grows.
- Reporting logic becomes difficult to maintain.

A dimensional data warehouse solves these challenges by organizing business data into Facts and Dimensions.

# Business Process Identification

Before designing any warehouse, we must identify the primary business processes.

For this project, the core business processes include:

- Customer purchases
- Order fulfillment
- Product sales
- Payments
- Customer reviews
- Seller performance
- Logistics and delivery

Each business process generates measurable events that will later become Fact Tables.

# Defining the Grain

The grain defines exactly what one row in a fact table represents.

A clearly defined grain ensures consistency across every analytical metric.

For this project, the primary grain will be:

One row represents one purchased product within a customer order.

This grain allows accurate analysis of:

- Revenue
- Quantity sold
- Product performance
- Customer purchasing behavior
- Seller performance
- Delivery efficiency

# Star Schema Overview

The analytical warehouse will use a Star Schema.

The central fact table will store measurable sales events, while dimension tables will provide descriptive context such as customer, product, seller, date, and location.

The proposed structure is:

- `fact_sales`
- `dim_customer`
- `dim_product`
- `dim_seller`
- `dim_date`
- `dim_location`

Supporting fact tables will be introduced for processes that have a different grain:

- `fact_payments`
- `fact_reviews`
- `fact_returns` (synthetic)

# Fact Table: fact_sales

## Business Purpose

Stores product-level sales events for customer orders.

## Grain

One row represents one purchased product within one customer order.

## Source Tables

- `olist_order_items_dataset`
- `olist_orders_dataset`
- `olist_customers_dataset`

## Proposed Keys

- `sales_key` — surrogate primary key
- `order_id` — source business identifier
- `order_item_id` — sequence of the item within the order
- `customer_key` — foreign key to `dim_customer`
- `product_key` — foreign key to `dim_product`
- `seller_key` — foreign key to `dim_seller`
- `purchase_date_key` — foreign key to `dim_date`
- `delivery_date_key` — foreign key to `dim_date`
- `customer_location_key` — foreign key to `dim_location`
- `seller_location_key` — foreign key to `dim_location`

## Measures

- `item_price`
- `freight_value`
- `gross_item_value`
- `quantity`
- `delivery_days`
- `delivery_delay_days`
- `is_delayed`

# Dimension Table: dim_customer

## Business Purpose

Stores customer identity and geographic attributes used for customer analysis.

## Grain

One row represents one unique customer.

## Source Table

- `olist_customers_dataset`

## Proposed Columns

- `customer_key` — surrogate primary key
- `customer_unique_id` — durable source identifier
- `customer_city`
- `customer_state`
- `customer_zip_code_prefix`
- `customer_segment` — synthetic
- `acquisition_channel` — synthetic
- `registration_date` — synthetic
- `loyalty_status` — synthetic

# Dimension Table: dim_product

## Business Purpose

Stores descriptive and physical product attributes.

## Grain

One row represents one product.

## Source Tables

- `olist_products_dataset`
- `product_category_name_translation`

## Proposed Columns

- `product_key` — surrogate primary key
- `product_id` — source business identifier
- `category_name_portuguese`
- `category_name_english`
- `product_name_length`
- `product_description_length`
- `product_photos_qty`
- `product_weight_g`
- `product_length_cm`
- `product_height_cm`
- `product_width_cm`
- `brand_name` — synthetic
- `cost_price` — synthetic
- `launch_date` — synthetic

# Dimension Table: dim_seller

## Business Purpose

Stores seller identity, location, and marketplace attributes.

## Grain

One row represents one seller.

## Source Table

- `olist_sellers_dataset`

## Proposed Columns

- `seller_key` — surrogate primary key
- `seller_id` — source business identifier
- `seller_city`
- `seller_state`
- `seller_zip_code_prefix`
- `seller_onboarding_date` — synthetic
- `seller_status` — synthetic
- `seller_tier` — synthetic

# Dimension Table: dim_date

## Business Purpose

Provides reusable calendar attributes for time-based reporting.

## Grain

One row represents one calendar date.

## Proposed Columns

- `date_key` — integer key in YYYYMMDD format
- `full_date`
- `day`
- `day_name`
- `week_number`
- `month`
- `month_name`
- `quarter`
- `year`
- `is_weekend`

# Dimension Table: dim_location

## Business Purpose

Provides standardized geographic attributes for customers, sellers, and future warehouses.

## Grain

One row represents one ZIP-code prefix.

## Source Table

- Cleaned `olist_geolocation_dataset`

## Proposed Columns

- `location_key` — surrogate primary key
- `zip_code_prefix`
- `city`
- `state`
- `latitude`
- `longitude`
- `region`

# Fact Table: fact_payments

## Business Purpose

Stores payment transactions associated with customer orders.

## Grain

One row represents one payment transaction for one customer order.

## Source Table

- `olist_order_payments_dataset`

## Proposed Keys

- payment_key — surrogate primary key
- order_id — business identifier
- customer_key — foreign key
- payment_date_key — synthetic date key

## Measures

- payment_value
- payment_installments

## Attributes

- payment_type
- payment_sequence

# Fact Table: fact_reviews

## Business Purpose

Stores customer review information to support customer satisfaction analysis, product quality assessment, seller performance monitoring, and service quality reporting.

---

## Grain

One row represents one review record associated with one customer order.

The grain is defined at the review-record level rather than the review identifier because the source dataset allows a single `review_id` to be associated with multiple customer orders.

---

## Source Table

- `olist_order_reviews_dataset`

---

## Proposed Keys

### Primary Key

- `review_key` — Surrogate primary key generated during the ETL process.

### Business Identifiers

- `review_id`
- `order_id`

### Foreign Keys

- `customer_key`
- `review_date_key`

---

## Measures

- `review_score`

---

## Attributes

- `review_title`
- `review_message`
- `review_creation_date`
- `review_answer_timestamp`

---

## Primary Key Strategy

During the Validation phase, the `review_id` column was investigated after duplicate values were detected.

Validation findings showed:

- 789 duplicated review identifiers
- 1,603 total records associated with those identifiers
- identical review information across duplicated records
- different `order_id` values associated with the same `review_id`

This indicates that the source dataset intentionally allows a single review identifier to reference multiple customer orders.

As a result, `review_id` cannot be used as the warehouse primary key.

Instead, the analytical warehouse will generate a surrogate primary key (`review_key`) while preserving `review_id` as the original business identifier for traceability and auditing purposes.

----

## Business Value

The `fact_reviews` table enables analysis of:

- Customer satisfaction trends
- Product quality ratings
- Seller performance
- Review score distribution
- Customer feedback behaviour
- Review response timelines
- Service quality monitoring

# Fact Table: fact_returns (Synthetic)

## Business Purpose

Stores returned products and refund information.

## Grain

One row represents one returned product.

## Source

Synthetic dataset generated for analytical purposes.

## Proposed Keys

- return_key
- sales_key
- customer_key
- product_key
- seller_key
- return_date_key

## Measures

- refund_amount
- return_quantity

## Attributes

- return_reason
- return_status

# Data Warehouse Loading Strategy

The analytical warehouse will be populated through an ETL (Extract, Transform, Load) pipeline.

Each table will be loaded in a specific sequence to maintain referential integrity and ensure that all foreign keys reference existing dimension records.

The loading process follows a dimension-first strategy, where all dimensions are populated before loading the fact tables.

# ETL Loading Order

The proposed loading sequence is:

1. dim_date
2. dim_location
3. dim_customer
4. dim_product
5. dim_seller
6. fact_sales
7. fact_payments
8. fact_reviews
9. fact_returns

# ETL Architecture

The ETL pipeline consists of three layers.

## Raw Layer

Stores the original Olist datasets exactly as received.

## Staging Layer

Applies data cleaning, standardization, type conversion, and quality checks.

## Warehouse Layer

Loads cleaned and transformed data into the dimensional model using surrogate keys.

# Data Flow

Raw Olist Data

↓

Staging Layer

↓

Dimension Tables

↓

Fact Tables

↓

Power BI

↓

Business Reports

↓

Executive Decision Making

# Slowly Changing Dimensions

Certain descriptive attributes may change over time.

Examples include:

- Customer loyalty status
- Customer segment
- Seller tier
- Seller status

For this project, the warehouse will initially implement Slowly Changing Dimension Type 1 (overwrite updates).

Future versions may implement Type 2 to preserve historical changes.

# Enterprise Data Quality Framework

The purpose of the data quality framework is to ensure that only accurate, complete, consistent, and reliable data enters the analytical warehouse.

The following validation rules will be applied during the staging and warehouse loading processes.

## 1. Primary Key Validation

- Primary keys must not contain missing values.
- Primary keys must be unique within their respective tables.
- Duplicate business records must be investigated before loading.

## 2. Foreign Key Validation

- Every foreign key must reference an existing record in the related dimension or fact table.
- Records with unmatched foreign keys must be flagged for investigation.
- Fact tables must not be loaded until the required dimension records exist.

## 3. Missing Value Validation

- Missing values must be reviewed according to their business meaning.
- Expected missing values, such as absent review comments, may be retained.
- Critical missing identifiers must prevent the affected record from being loaded.
- Missing operational timestamps must be investigated using related status fields.

## 4. Duplicate Validation

- Exact duplicate rows must be identified.
- Duplicate business keys must be investigated based on the table grain.
- Repeated identifiers that represent valid one-to-many relationships must not be incorrectly removed.

## 5. Data Type Validation

- Date fields must be converted to valid date or timestamp formats.
- Monetary values must use numeric data types.
- Identifier fields must use consistent text formats.
- Boolean indicators must use standardized values.

## 6. Business Rule Validation

- Payment values must be greater than zero.
- Product prices and freight values must not be negative.
- Review scores must fall within the accepted rating range.
- Delivery dates must not occur before purchase dates.
- Return quantities must be greater than zero.
- Every returned item must reference an existing sales record.

## 7. Referential Integrity

- Customer, product, seller, date, and location keys must exist before fact records are loaded.
- Orphan records must be logged and excluded from the warehouse until resolved.

## 8. Data Quality Logging

The ETL pipeline will record:

- number of source rows,
- number of successfully loaded rows,
- number of rejected rows,
- duplicate counts,
- missing-value counts,
- foreign-key failures,
- and validation errors.

## 9. Error Handling

Invalid records will not be silently discarded.

They will be:

1. flagged,
2. logged,
3. stored separately for review,
4. and corrected or excluded based on documented business rules.

## 10. Quality Objective

The warehouse should provide consistent and trustworthy data for:

- SQL analysis,
- Power BI dashboards,
- business reporting,
- statistical analysis,
- and future machine-learning applications.